# Определение автоматизированного трафика Avito

## Подход текстом

**Задача.** Каждой `cookie_id` из теста нужно присвоить скор 0..1 (вероятность бота); метрика — precision при recall ≥ 70%.

**Ключевая идея.** Единица модели — куки, а не событие. У каждой куки есть индивидуальное 24-часовое окно `[window_start_ts, window_end_ts)`, и признаки строятся только из событий внутри него: события вне окна описывают уже другой период (а `captcha_shown` — вообще всегда происходит после конца окна, то есть является следствием блокировки и подглядыванием в будущее). Корректность этого фильтра проверяется тестом `extras/test_window_leakage.py`.

**Признаки** (347 на куки): счётчики и доли типов событий, платформ (с нормализацией регистра), категорий, локаций, продавцов; разнообразие и энтропии распределений; статистики межсобытийных интервалов и темпа; сессии (граница 30 минут); распределение по часам; путь и скорость курсора между наблюдаемыми точками; переходы между соседними событиями; навигация по страницам поиска. Точные дубликаты строк событий (4863 шт.) удаляются при загрузке.

**Модель.** Финальный кандидат — одиночный `CatBoostClassifier` с весами классов SqrtBalanced, усреднённый по мешку из 7 сидов. Ансамбли и OOF-стекинг сравнивались честно (13 кандидатов), но устойчивого преимущества на отборочных фолдах не дали.

**Валидация.** Случайные фолды для временной задачи не подходят: тест идёт 7 днями сразу после train. Поэтому решения принимаются на двух хронологических SELECT-фолдах (expanding window), честное число получается на нетронутом EXAM-блоке 17.04 по 19.04, а репрезентативная оценка для теста — на HORIZON-фолде формы «7 дней обучения → 7 дней предсказания»: **P@R70 = 0,7566, 95% CI [0,68; 0,84]**. Все кандидаты сравниваются как мешки из 5 сидов, потому что разброс одной модели между сидами (±2 п.п.) сопоставим с различием моделей.

**Воспроизводимость.** Вся случайность зафиксирована в `src/config.py`; одна команда `python run.py` воспроизводит `submission.csv`. Подробности, таблицы результатов и отрицательные эксперименты — в `REPORT.md`.

## Запуск пайплайна

Одна команда строит признаки, выполняет валидацию SELECT → EXAM → HORIZON, переобучает выбранный рецепт и пишет `submission.csv`. Полный прогон — от 1 до 1,5 часов на 22-ядерном CPU. Проверенная версия Python: 3.12.4; зависимости: `python -m pip install -r requirements.txt`.

Что будет происходить внизу (по строкам лога):

1. загрузка данных и удаление точных дубликатов событий;
2. построение признаков (около минуты);
3. SELECT: подбор весов классов для каждой модели, проверка полного набора признаков против топ-60, таблица из 13 кандидатов;
4. EXAM и HORIZON: честные оценки замороженного рецепта на нетронутых блоках, с интервалами и профилем ошибок;
5. финальное обучение победителя на всём train и запись `submission.csv`.

In [ ]:
from pathlib import Path
import subprocess
import sys

project_dir = Path.cwd()
# Запускаю через subprocess, чтобы лог шёл в вывод ячейки построчно.
# check=True: если пайплайн упадёт, ноутбук тоже упадёт, а не продолжит
# молча работать со старым submission.csv.
subprocess.run(
    [sys.executable, str(project_dir / "run.py")],
    cwd=project_dir,
    check=True,
)


## Проверка файла отправки

Контракт сабмита проверяется и при записи (`src/train.py`), а отдельный скрипт `check_submission.py` перепроверяет готовый файл, включая подсчёт одинаковых score (метрика склеивает их в одну группу). Эта ячейка дублирует базовые инварианты прямо в ноутбуке, чтобы не открывать терминал.

In [ ]:
import pandas as pd

test = pd.read_csv(project_dir / "data" / "test.csv", usecols=["cookie_id"])
submission = pd.read_csv(project_dir / "submission.csv")

# Ровно две колонки, одна строка на тестовую куку, тот же порядок,
# скоры конечные и в [0, 1]. Если что-то из этого не так, платформа
# файл просто не примет, лучше узнать об этом здесь.
assert submission.columns.tolist() == ["cookie_id", "score"]
assert len(submission) == len(test)
assert submission["cookie_id"].is_unique
assert submission["cookie_id"].equals(test["cookie_id"])
assert submission["score"].between(0, 1).all()
print(submission.shape)
submission.head()
